In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import KFold, StratifiedKFold, cross_val_score
from sklearn import linear_model, tree, ensemble

import os
for dirname, _, filenames in os.walk('kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

In [ ]:
train_data = pd.read_csv('kaggle/input/train.csv')
# pd.set_option('display.max_columns', None)
# print(train_data.head())

print(train_data.columns.tolist())
print(train_data.info())

In [ ]:
train_data = pd.read_csv('kaggle/input/train.csv')

train_data.dropna(axis=0, subset=['Survived'], inplace=True)
y = train_data.Survived # Target varible
train_data.drop(['Survived'], axis=1, inplace=True) # Removing target variable from training data

train_data.drop(['Age'], axis=1, inplace=True) # Remove columns with null values

# Select numeric columns only
numeric_cols = [cname for cname in train_data.columns
                if train_data[cname].dtype in ['int64', 'float64'] and cname != 'PassengerId']

# print(numeric_cols)
# print(type(numeric_cols))
# arr_1d = np.array(numeric_cols)
# print(type(arr_1d))
# print(arr_1d.shape)
# matrix = arr_1d.reshape(1, 5)
# print(matrix.shape)
# print(matrix)

X = train_data[numeric_cols].copy()

print("Shape of input data: {} and shape of target variable: {}".format(X.shape, y.shape))
pd.concat([X, y], axis=1).head() # Show first 5 training examples

In [ ]:
kf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

cnt = 1

for train_index, test_index in kf.split(X, y):
    print(f"Fold:{cnt}, Train set: {len(train_index)}, Test set: {len(test_index)}")
    cnt += 1

In [ ]:
# 이 코드 한 줄 안에는 모델 학습, 예측, 그리고 채점 과정이 한 방에 압축되어 있습니다.
score = cross_val_score(tree.DecisionTreeClassifier(random_state=42),
                        X, y, cv=kf, scoring="accuracy")
print(f"Scores for each fold are: {score}")
print(f"Average score: {score.mean():.2f}")

In [ ]:
score = cross_val_score(ensemble.RandomForestClassifier(random_state=42),
                        X, y, cv=kf, scoring="accuracy")
print(f"Scores for each fold are: {score}")
print(f"Average score: {'{:.2f}'.format(score.mean())}")

In [ ]:
final_model = ensemble.RandomForestClassifier(random_state=42)
final_model.fit(X, y)

importances = final_model.feature_importances_

feat_importances = pd.Series(importances, index=X.columns)
print(feat_importances.sort_values(ascending=False))

In [ ]:
test_data = pd.read_csv('kaggle/input/test.csv')
# print(test_data.head())
test_data.drop(['Age'], axis=1, inplace=True)

numeric_cols = [cname for cname in test_data.columns
                if test_data[cname].dtype in ['float64', 'int64'] and cname != 'PassengerId']

x = test_data[numeric_cols].copy()
x.head()

In [ ]:
final_model = ensemble.RandomForestClassifier(random_state=42)

final_model.fit(X, y)

In [ ]:
predictions = final_model.predict(x)

In [ ]:
print(predictions[:10])

In [ ]:
submission = pd.DataFrame({
    "PassengerId": test_data["PassengerId"],
    "Survived": predictions
})
submission.to_csv('submission.csv', index=False)

result = pd.read_csv("submission.csv")
result